In [1]:
import pandas as pd
df=pd.read_csv("Downloads/dirty_cafe_sales.csv")

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [3]:
#=============================================
# HEADER CLEANING#
#=============================================

In [4]:
df.columns=df.columns.str.strip().str.lower().str.replace(" ","_")
df.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [5]:
#=============================================
# FIXING THE DATATYPES
#=============================================

In [6]:
df["item"]=df["item"].astype("string",errors="ignore")

In [7]:
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["price_per_unit"] = pd.to_numeric(df["price_per_unit"], errors="coerce")
df["total_spent"] = pd.to_numeric(df["total_spent"], errors="coerce")

In [8]:
df["transaction_date"]=pd.to_datetime(df["transaction_date"],errors='coerce')

In [9]:
df["payment_method"]=df["payment_method"].astype("string",errors="ignore")
df["location"]=df["location"].astype("string",errors="ignore")


In [10]:
df.dtypes

transaction_id              object
item                string[python]
quantity                   float64
price_per_unit             float64
total_spent                float64
payment_method      string[python]
location            string[python]
transaction_date    datetime64[ns]
dtype: object

In [11]:
#=============================================================
# FIXLLING THE NaN OF "Item","Quantity","Price","Total_Spent"
#=============================================================

In [12]:
df["total_spent"]=df["total_spent"].fillna(df["quantity"]*df["price_per_unit"])

In [13]:
df["price_per_unit"]=df["price_per_unit"].fillna(df["total_spent"]/df["quantity"])

In [14]:
df["quantity"]=df["quantity"].fillna(df["total_spent"]/df["price_per_unit"])

In [15]:
#df.head(100)

In [16]:
#=========================================
#CONVERTING IRRELEVANT VALUES TO NaN
#=========================================

In [17]:
df["item"].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',       <NA>,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: string

In [18]:
df["item"]=df["item"].replace(["ERROR","UNKNOWN"],pd.NA)
df["payment_method"]=df["payment_method"].replace(["ERROR","UNKNOWN"],pd.NA)
df["location"]=df["location"].replace(["ERROR","UNKNOWN"],pd.NA)




In [19]:
#FILLING NULL VALUES OF PRICE_PER_UNIT USING DICT MAPPING

In [20]:
item_to_price={"Coffee":2.0 , "Cake":3.0,"Cookie":1.0,"Salad":5.0,"Smoothie":4.0,"Sandwich":4.0, "Juice":3.0,"Tea":1.5,}

In [21]:
df["price_per_unit"]=df["price_per_unit"].fillna(df["item"].map(item_to_price))

In [22]:
df["price_per_unit"].isna().sum()

np.int64(6)

In [23]:
#FILLING NULL VALUES OF ITEMS USING REVERSE MAPPING

In [24]:
price_to_item={v: k for k,v in item_to_price.items()}
print(price_to_item)

{2.0: 'Coffee', 3.0: 'Juice', 1.0: 'Cookie', 5.0: 'Salad', 4.0: 'Sandwich', 1.5: 'Tea'}


In [25]:
df["item"]=df["item"].fillna(df["price_per_unit"].map(price_to_item))

In [26]:
#=============================================================
#HANDLING ROWS WITH MULTIPLE NaN VALUES(PAYMENT_METHOD AND LOCATION)
#=============================================================

In [27]:
df=df.dropna(subset=["price_per_unit"])

In [28]:
df["payment_method"]=df["payment_method"].fillna("Unknown")
    

In [29]:
df["location"]=df["location"].fillna("Unknown")

In [30]:
#==============================================================
#HANDLING MISSING VALUES OF DATE COLUMN USING FFILL(FORWARD FILL)
#==============================================================

In [31]:
df["transaction_date"].isna().sum()

np.int64(460)

In [32]:
df = df.sort_values("transaction_date")

In [33]:
df["transaction_date"]=df["transaction_date"].fillna(method="ffill")

C:\Users\prath\AppData\Local\Temp\ipykernel_500\850443961.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["transaction_date"]=df["transaction_date"].fillna(method="ffill")


In [34]:
df.tail(34)

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
9325,TXN_4962500,Juice,5.0,3.0,15.0,Unknown,Takeaway,2023-12-31
9372,TXN_4003831,Cookie,1.0,1.0,1.0,Credit Card,Unknown,2023-12-31
9380,TXN_9877722,Sandwich,5.0,4.0,20.0,Digital Wallet,In-store,2023-12-31
9382,TXN_4255580,Juice,1.0,3.0,3.0,Cash,Unknown,2023-12-31
9404,TXN_1526920,Sandwich,5.0,4.0,20.0,Digital Wallet,Takeaway,2023-12-31
9411,TXN_9663941,Juice,4.0,3.0,12.0,Cash,In-store,2023-12-31
9428,TXN_4635386,Tea,1.0,1.5,1.5,Credit Card,Takeaway,2023-12-31
9451,TXN_9055374,Sandwich,1.0,4.0,4.0,Unknown,Takeaway,2023-12-31
9454,TXN_1637131,Tea,4.0,1.5,6.0,Cash,Unknown,2023-12-31
9457,TXN_1852436,Juice,4.0,3.0,12.0,Unknown,Unknown,2023-12-31


In [35]:
#===========================================================================
#FINALLY DROPPING THE ROWS WHERE QUANTITY AND TOTAL BOTH ARE NaN
#===========================================================================

In [36]:
df.isna().sum()

transaction_id       0
item                 0
quantity            35
price_per_unit       0
total_spent         37
payment_method       0
location             0
transaction_date     0
dtype: int64

In [37]:
df=df.dropna(subset="quantity")

In [38]:
#===========================================================================
#DATA CLEANING COMPLETE
#NOW SAVING IT 
#===========================================================================

In [41]:
df.to_csv(r"C:\Users\prath\OneDrive\Documents\PRACTISE STUFF\cleaned_data.csv", index=False)